# 🚀 Churn Prediction: Advanced Stacked Ensemble Pipeline (Rule v2)

Notebook này triển khai quy trình mô hình hóa nâng cao với **Kỹ thuật Ensemble Stacking** kết hợp 3 thuật toán nền tảng mạnh mẽ nhất cho bài toán Churn Tabular:
1. **Logistic Regression (L2 Regularized + Standardized)** — Baseline tuyến tính chuẩn mực
2. **LightGBM (GBDT)** — Tối ưu hóa siêu tham số bằng Grid Search theo tiêu chí **AUCPR (PR-AUC)**
3. **XGBoost (Extreme Gradient Boosting)** — Mô hình cây quyết định với kiểm soát Regularization L1/L2
4. **Stacked Ensemble Model** — Kết hợp dự đoán của cả 3 mô hình thông qua **Meta-Learner** được huấn luyện trên **5-Fold Cross-Validation Out-Of-Fold (OOF)**

---

### 📌 Quy trình thực hiện:
```
[Tập Dữ Liệu v2] ──> [Chronological Split] ──> [Train / Val / Test]
                            │
              ┌─────────────┴─────────────┐
              ▼                           ▼
  [Grid Search GBM (AUCPR)]     [5-Fold CV trên Train]
              │                           │
              └─────────────┬─────────────┘
                            ▼
           [Out-Of-Fold (OOF) Predictions]
                            │
                            ▼
           [Huấn luyện Meta-Learner (Stacking)]
                            │
                            ▼
           [Retrain 3 Base Models trên Full Train]
                            │
                            ▼
           [Đánh giá 4 Mô hình trên Test Set] ──> [Chọn Best Model theo AUCPR > AUC]
                            │
                            ▼
           [Feature Importance (GBM) + Lưu Artifact Model Bundle]
```


## 1. Nạp Thư viện & Thiết lập Cấu hình


In [1]:
import warnings, datetime, logging, sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import xgboost as xgb
import sklearn
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    roc_curve,
    precision_recall_curve
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# ── Hằng số cấu hình ──────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT / "output"
ART_DIR = ROOT / "artifacts"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
ART_DIR.mkdir(exist_ok=True, parents=True)

# ── Khoảng thời gian phân chia (Chronological Split) ───────────────────
TRAIN_START = pd.Timestamp("2024-09-01")
TRAIN_END   = pd.Timestamp("2025-08-01")
VAL_START   = pd.Timestamp("2025-09-01")
VAL_END     = pd.Timestamp("2026-02-01")
TEST_START  = pd.Timestamp("2026-03-01")
TEST_END    = pd.Timestamp("2026-06-01")

print("=" * 60)
print("PHIÊN BẢN CÁC THƯ VIỆN SỬ DỤNG:")
print(f"  Python       : {sys.version.split()[0]}")
print(f"  LightGBM     : {lgb.__version__}")
print(f"  XGBoost      : {xgb.__version__}")
print(f"  Scikit-Learn : {sklearn.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  NumPy        : {np.__version__}")
print("=" * 60)


PHIÊN BẢN CÁC THƯ VIỆN SỬ DỤNG:
  Python       : 3.11.9
  LightGBM     : 4.7.0
  XGBoost      : 3.2.0
  Scikit-Learn : 1.9.0
  Pandas       : 3.0.5
  NumPy        : 2.4.6


## 2. Đọc Dữ liệu Dataset v2 & Lọc Đặc trưng


In [2]:
data_path = OUTPUT_DIR / "churn_temporal_dataset_v2.parquet"
if not data_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {data_path}")

df = pd.read_parquet(data_path)
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

print(f"Kích thước tập dữ liệu : {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Phạm vi thời gian       : {df['snapshot_date'].min().date()} → {df['snapshot_date'].max().date()}")

# ── Danh sách cột Meta/Nhãn — TUYỆT ĐỐI không dùng làm đặc trưng ─────
META_COLS = {
    "customer_id", "snapshot_date", "churn_next_30d",
    "rule1_closed", "rule2_downgrade_to_free_inactive",
    "rule3_free_at_snapshot_inactive",
    "churn_reason", "tier_at_snapshot", "closed_date", "label_complete",
}

feature_cols = [
    c for c in df.columns
    if c not in META_COLS
    and not c.startswith("future_")
    and not c.endswith("_future")
]

# Kiểm tra an toàn chống rò rỉ nhãn
leaked = [c for c in feature_cols if any(k in c.lower() for k in ["rule", "future", "closed", "churn"])]
if leaked:
    print(f"⚠️ Phát hiện và loại bỏ cột rò rỉ: {leaked}")
    feature_cols = [c for c in feature_cols if c not in leaked]

print(f"Tổng số đặc trưng hợp lệ đưa vào huấn luyện: {len(feature_cols)}")
print(f"Cột nhãn mục tiêu: churn_next_30d")


Kích thước tập dữ liệu : 185,160 dòng × 387 cột
Phạm vi thời gian       : 2023-08-01 → 2026-08-01
Tổng số đặc trưng hợp lệ đưa vào huấn luyện: 379
Cột nhãn mục tiêu: churn_next_30d


## 3. Phân chia Dữ liệu theo Dòng thời gian (Chronological Split)


In [3]:
train_mask = (df["snapshot_date"] >= TRAIN_START) & (df["snapshot_date"] <= TRAIN_END)
val_mask   = (df["snapshot_date"] >= VAL_START)   & (df["snapshot_date"] <= VAL_END)
test_mask  = (df["snapshot_date"] >= TEST_START)  & (df["snapshot_date"] <= TEST_END)

X_train_raw = df.loc[train_mask, feature_cols].copy()
y_train = df.loc[train_mask, "churn_next_30d"].to_numpy().astype(np.int32)

X_val_raw   = df.loc[val_mask, feature_cols].copy()
y_val   = df.loc[val_mask, "churn_next_30d"].to_numpy().astype(np.int32)

X_test_raw  = df.loc[test_mask, feature_cols].copy()
y_test  = df.loc[test_mask, "churn_next_30d"].to_numpy().astype(np.int32)

splits_info = [
    ("Train",      TRAIN_START, TRAIN_END, y_train),
    ("Validation", VAL_START,   VAL_END,   y_val),
    ("Test",       TEST_START,  TEST_END,  y_test),
]

print("=" * 72)
print(f"  {'Tập':<12} {'Khoảng thời gian':<28} {'Số dòng':>8} {'Churn':>8} {'Tỷ lệ Churn':>12}")
print("-" * 72)
for name, s, e, y in splits_info:
    n, c = len(y), int(y.sum())
    rate = c / n if n > 0 else 0
    print(f"  {name:<12} {str(s.date()) + ' → ' + str(e.date()):<28} {n:>8,} {c:>8,} {rate:>11.2%}")
print("=" * 72)


  Tập          Khoảng thời gian              Số dòng    Churn  Tỷ lệ Churn
------------------------------------------------------------------------
  Train        2024-09-01 → 2025-08-01        62,729   17,214      27.44%
  Validation   2025-09-01 → 2026-02-01        45,586   10,844      23.79%
  Test         2026-03-01 → 2026-06-01        35,336    3,486       9.87%


## 4. Tiền xử lý Dữ liệu: Imputation & Scaling
- **SimpleImputer (strategy='median')**: Điền giá trị thiếu (fit **chỉ trên tập Train** để chống data leakage)
- **StandardScaler**: Chuẩn hóa Z-score cần thiết cho Logistic Regression


In [4]:
# 1. Điền giá trị thiếu bằng trung vị fit trên Train
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp   = imputer.transform(X_val_raw)
X_test_imp  = imputer.transform(X_test_raw)

# 2. Chuẩn hóa đặc trưng (dành riêng cho Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled   = scaler.transform(X_val_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print(f"Imputation & Scaling hoàn tất:")
print(f"  Kích thước X_train: {X_train_imp.shape}")
print(f"  Kích thước X_val  : {X_val_imp.shape}")
print(f"  Kích thước X_test : {X_test_imp.shape}")
print(f"  Số giá trị NaN sau xử lý: {np.isnan(X_train_imp).sum() + np.isnan(X_val_imp).sum() + np.isnan(X_test_imp).sum()}")


Imputation & Scaling hoàn tất:
  Kích thước X_train: (62729, 379)
  Kích thước X_val  : (45586, 379)
  Kích thước X_test : (35336, 379)
  Số giá trị NaN sau xử lý: 0


## 5. Grid Search Tinh chỉnh Siêu tham số LightGBM theo AUCPR (PR-AUC)
Thử nghiệm các tổ hợp hyperparameter chính và chọn cấu hình có **Validation AUCPR** cao nhất.


In [5]:
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"Hệ số cân bằng lớp (scale_pos_weight) = {scale_pos_weight:.2f}")

# Lưới tham số tìm kiếm
param_grid = [
    {"num_leaves": 31, "max_depth": 5, "learning_rate": 0.05, "n_estimators": 400, "colsample_bytree": 0.8, "subsample": 0.8, "reg_alpha": 0.5, "reg_lambda": 1.0},
    {"num_leaves": 63, "max_depth": 6, "learning_rate": 0.03, "n_estimators": 500, "colsample_bytree": 0.8, "subsample": 0.8, "reg_alpha": 1.0, "reg_lambda": 2.0},
    {"num_leaves": 31, "max_depth": 4, "learning_rate": 0.03, "n_estimators": 500, "colsample_bytree": 0.7, "subsample": 0.8, "reg_alpha": 0.1, "reg_lambda": 1.0},
    {"num_leaves": 45, "max_depth": 5, "learning_rate": 0.05, "n_estimators": 450, "colsample_bytree": 0.85, "subsample": 0.85, "reg_alpha": 0.5, "reg_lambda": 1.5},
]

print("BẮT ĐẦU GRID SEARCH LIGHTGBM (ĐÁNH GIÁ THEO VAL AUCPR)...")
print("-" * 75)

best_lgb_aucpr = -1.0
best_lgb_params = None
grid_results = []

for i, p in enumerate(param_grid, 1):
    clf_tune = lgb.LGBMClassifier(
        random_state=SEED,
        scale_pos_weight=scale_pos_weight,
        verbose=-1,
        **p
    )
    clf_tune.fit(X_train_imp, y_train)
    val_probs_tune = clf_tune.predict_proba(X_val_imp)[:, 1]
    
    val_aucpr = average_precision_score(y_val, val_probs_tune)
    val_roc_auc = roc_auc_score(y_val, val_probs_tune)
    
    grid_results.append({
        "config_id": i,
        "params": p,
        "val_AUCPR": val_aucpr,
        "val_ROC_AUC": val_roc_auc
    })
    
    print(f"  Config #{i:02d} | leaves={p['num_leaves']}, depth={p['max_depth']}, lr={p['learning_rate']} "
          f"--> Val AUCPR: {val_aucpr:.4f} | Val ROC-AUC: {val_roc_auc:.4f}")
    
    if val_aucpr > best_lgb_aucpr:
        best_lgb_aucpr = val_aucpr
        best_lgb_params = p

print("-" * 75)
print(f"⭐ CẤU HÌNH TỐI ƯU NHẤT CHO LIGHTGBM (Val AUCPR = {best_lgb_aucpr:.4f}):")
for k, v in best_lgb_params.items():
    print(f"    - {k}: {v}")


Hệ số cân bằng lớp (scale_pos_weight) = 2.64
BẮT ĐẦU GRID SEARCH LIGHTGBM (ĐÁNH GIÁ THEO VAL AUCPR)...
---------------------------------------------------------------------------
  Config #01 | leaves=31, depth=5, lr=0.05 --> Val AUCPR: 0.5435 | Val ROC-AUC: 0.8576
  Config #02 | leaves=63, depth=6, lr=0.03 --> Val AUCPR: 0.5421 | Val ROC-AUC: 0.8569
  Config #03 | leaves=31, depth=4, lr=0.03 --> Val AUCPR: 0.5421 | Val ROC-AUC: 0.8581
  Config #04 | leaves=45, depth=5, lr=0.05 --> Val AUCPR: 0.5422 | Val ROC-AUC: 0.8562
---------------------------------------------------------------------------
⭐ CẤU HÌNH TỐI ƯU NHẤT CHO LIGHTGBM (Val AUCPR = 0.5435):
    - num_leaves: 31
    - max_depth: 5
    - learning_rate: 0.05
    - n_estimators: 400
    - colsample_bytree: 0.8
    - subsample: 0.8
    - reg_alpha: 0.5
    - reg_lambda: 1.0


## 6. Khởi tạo 3 Mô hình Cơ sở (Base Models)


In [6]:
# 1. Logistic Regression
model_lr = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=SEED,
    n_jobs=-1
)

# 2. LightGBM (với tham số tối ưu từ Grid Search)
model_lgb = lgb.LGBMClassifier(
    random_state=SEED,
    scale_pos_weight=scale_pos_weight,
    verbose=-1,
    **best_lgb_params
)

# 3. XGBoost
model_xgb = xgb.XGBClassifier(
    random_state=SEED,
    scale_pos_weight=scale_pos_weight,
    n_estimators=450,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="logloss"
)

print("Đã khởi tạo xong 3 mô hình cơ sở:")
print("  [1] Logistic Regression (L2)")
print(f"  [2] LightGBM (depth={best_lgb_params['max_depth']}, leaves={best_lgb_params['num_leaves']})")
print("  [3] XGBoost (depth=5, lr=0.03)")


Đã khởi tạo xong 3 mô hình cơ sở:
  [1] Logistic Regression (L2)
  [2] LightGBM (depth=5, leaves=31)
  [3] XGBoost (depth=5, lr=0.03)


## 7. 5-Fold Cross-Validation trên Train để tạo Out-Of-Fold (OOF) Predictions
Để huấn luyện Meta-Learner mà **không bị target leakage (overfitting)**, chúng ta sử dụng Stratified 5-Fold CV trên tập Train:
- Dự đoán xác suất Out-Of-Fold của từng mô hình sẽ tạo thành ma trận đặc trưng `[oof_lr, oof_lgb, oof_xgb]` cho Meta-Learner.


In [7]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lr  = np.zeros(len(X_train_imp), dtype=np.float32)
oof_lgb = np.zeros(len(X_train_imp), dtype=np.float32)
oof_xgb = np.zeros(len(X_train_imp), dtype=np.float32)

print(f"BẮT ĐẦU {N_SPLITS}-FOLD CROSS VALIDATION TRÊN TẬP TRAIN...")
print("=" * 70)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_imp, y_train), 1):
    # Dữ liệu cho fold hiện tại
    X_tr_imp, y_tr = X_train_imp[train_idx], y_train[train_idx]
    X_va_imp, y_va = X_train_imp[val_idx], y_train[val_idx]
    
    X_tr_sc, X_va_sc = X_train_scaled[train_idx], X_train_scaled[val_idx]
    
    # 1. Fit & Predict LR
    model_lr_fold = LogisticRegression(penalty="l2", C=1.0, max_iter=1000, solver="lbfgs", random_state=SEED)
    model_lr_fold.fit(X_tr_sc, y_tr)
    oof_lr[val_idx] = model_lr_fold.predict_proba(X_va_sc)[:, 1]
    
    # 2. Fit & Predict LightGBM
    model_lgb_fold = lgb.LGBMClassifier(random_state=SEED, scale_pos_weight=scale_pos_weight, verbose=-1, **best_lgb_params)
    model_lgb_fold.fit(X_tr_imp, y_tr)
    oof_lgb[val_idx] = model_lgb_fold.predict_proba(X_va_imp)[:, 1]
    
    # 3. Fit & Predict XGBoost
    model_xgb_fold = xgb.XGBClassifier(random_state=SEED, scale_pos_weight=scale_pos_weight, n_estimators=450,
                                       max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                                       tree_method="hist", eval_metric="logloss")
    model_xgb_fold.fit(X_tr_imp, y_tr)
    oof_xgb[val_idx] = model_xgb_fold.predict_proba(X_va_imp)[:, 1]
    
    fold_auc_lr  = average_precision_score(y_va, oof_lr[val_idx])
    fold_auc_lgb = average_precision_score(y_va, oof_lgb[val_idx])
    fold_auc_xgb = average_precision_score(y_va, oof_xgb[val_idx])
    
    print(f"  Fold {fold}/{N_SPLITS} | AUCPR -> LR: {fold_auc_lr:.4f} | LGB: {fold_auc_lgb:.4f} | XGB: {fold_auc_xgb:.4f}")

print("=" * 70)
print("KẾT QUẢ TỔNG HỢP OUT-OF-FOLD (OOF AUCPR TRÊN TRAIN):")
print(f"  Logistic Regression OOF AUCPR : {average_precision_score(y_train, oof_lr):.4f} (ROC-AUC: {roc_auc_score(y_train, oof_lr):.4f})")
print(f"  LightGBM            OOF AUCPR : {average_precision_score(y_train, oof_lgb):.4f} (ROC-AUC: {roc_auc_score(y_train, oof_lgb):.4f})")
print(f"  XGBoost             OOF AUCPR : {average_precision_score(y_train, oof_xgb):.4f} (ROC-AUC: {roc_auc_score(y_train, oof_xgb):.4f})")
print("=" * 70)


BẮT ĐẦU 5-FOLD CROSS VALIDATION TRÊN TẬP TRAIN...
  Fold 1/5 | AUCPR -> LR: 0.5466 | LGB: 0.5651 | XGB: 0.5664
  Fold 2/5 | AUCPR -> LR: 0.5626 | LGB: 0.5812 | XGB: 0.5775
  Fold 3/5 | AUCPR -> LR: 0.5420 | LGB: 0.5752 | XGB: 0.5713
  Fold 4/5 | AUCPR -> LR: 0.5605 | LGB: 0.5792 | XGB: 0.5767
  Fold 5/5 | AUCPR -> LR: 0.5492 | LGB: 0.5720 | XGB: 0.5712
KẾT QUẢ TỔNG HỢP OUT-OF-FOLD (OOF AUCPR TRÊN TRAIN):
  Logistic Regression OOF AUCPR : 0.5515 (ROC-AUC: 0.8299)
  LightGBM            OOF AUCPR : 0.5741 (ROC-AUC: 0.8384)
  XGBoost             OOF AUCPR : 0.5722 (ROC-AUC: 0.8386)


## 8. Huấn luyện Meta-Learner (Stacked Ensemble) & Retrain Base Models trên Full Train
1. Ma trận OOF Meta-features: `X_meta_train = [oof_lr, oof_lgb, oof_xgb]`
2. Huấn luyện Meta-Learner (Logistic Regression) trên `X_meta_train`
3. Retrain cả 3 Base Models trên toàn bộ tập Train (`X_train_imp` / `X_train_scaled`)
4. Sinh dự đoán trên Test set từ 3 mô hình và đưa qua Meta-Learner


In [8]:
# ── Bước 1: Huấn luyện Meta-Learner trên OOF Predictions ─────────────
X_meta_train = np.column_stack([oof_lr, oof_lgb, oof_xgb])

# Sử dụng Logistic Regression làm Meta-Learner
meta_learner = LogisticRegression(penalty="l2", C=1.0, max_iter=500, random_state=SEED)
meta_learner.fit(X_meta_train, y_train)

print("Huấn luyện Meta-Learner (Stacking) hoàn tất.")
print("Trọng số (Coefficients) của Meta-Learner đối với từng mô hình cơ sở:")
for name, coef in zip(["Logistic Regression", "LightGBM", "XGBoost"], meta_learner.coef_[0]):
    print(f"    - {name:<20}: {coef:.4f}")
print(f"    - Intercept           : {meta_learner.intercept_[0]:.4f}")

# ── Bước 2: Retrain các Base Models trên FULL TRAIN ──────────────────
print("\nĐang Retrain 3 Base Models trên toàn bộ dữ liệu Train...")
model_lr.fit(X_train_scaled, y_train)
model_lgb.fit(X_train_imp, y_train)
model_xgb.fit(X_train_imp, y_train)
print("Retrain hoàn tất!")

# ── Bước 3: Dự đoán trên Validation và Test Sets ─────────────────────
# Validation
val_p_lr  = model_lr.predict_proba(X_val_scaled)[:, 1]
val_p_lgb = model_lgb.predict_proba(X_val_imp)[:, 1]
val_p_xgb = model_xgb.predict_proba(X_val_imp)[:, 1]
X_meta_val = np.column_stack([val_p_lr, val_p_lgb, val_p_xgb])
val_p_ensemble = meta_learner.predict_proba(X_meta_val)[:, 1]

# Test
test_p_lr  = model_lr.predict_proba(X_test_scaled)[:, 1]
test_p_lgb = model_lgb.predict_proba(X_test_imp)[:, 1]
test_p_xgb = model_xgb.predict_proba(X_test_imp)[:, 1]
X_meta_test = np.column_stack([test_p_lr, test_p_lgb, test_p_xgb])
test_p_ensemble = meta_learner.predict_proba(X_meta_test)[:, 1]


Huấn luyện Meta-Learner (Stacking) hoàn tất.
Trọng số (Coefficients) của Meta-Learner đối với từng mô hình cơ sở:
    - Logistic Regression : -0.0574
    - LightGBM            : 2.7103
    - XGBoost             : 2.9058
    - Intercept           : -3.9483

Đang Retrain 3 Base Models trên toàn bộ dữ liệu Train...
Retrain hoàn tất!


## 9. Đánh giá & So sánh Toàn diện 4 Thuật toán trên Tập Test
Tiêu chuẩn chọn mô hình chiến thắng (**Best Model**):
- **Ưu tiên 1 (Primary)**: **Test AUCPR (PR-AUC)** cao nhất
- **Ưu tiên 2 (Tie-breaker)**: Nếu hòa AUCPR, chọn **Test ROC-AUC** cao nhất


In [9]:
# Hàm tìm threshold tối ưu trên Validation và đánh giá toàn diện
def evaluate_model_pipeline(name, val_probs, test_probs, y_val, y_test):
    # 1. Tìm threshold tối ưu theo F1 trên Validation
    best_f1, best_th = -1.0, 0.5
    for th in np.linspace(0.01, 0.99, 99):
        p_bin = (val_probs >= th).astype(int)
        f1_v = f1_score(y_val, p_bin, zero_division=0)
        if f1_v > best_f1:
            best_f1, best_th = f1_v, float(th)
            
    # 2. Đánh giá trên tập Test với ngưỡng tối ưu
    test_preds = (test_probs >= best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
    
    aucpr = average_precision_score(y_test, test_probs)
    roc_auc = roc_auc_score(y_test, test_probs)
    prec = precision_score(y_test, test_preds, zero_division=0)
    rec = recall_score(y_test, test_preds, zero_division=0)
    f1_t = f1_score(y_test, test_preds, zero_division=0)
    brier = brier_score_loss(y_test, test_probs)
    
    return {
        "Model": name,
        "AUCPR": aucpr,
        "ROC_AUC": roc_auc,
        "F1": f1_t,
        "Precision": prec,
        "Recall": rec,
        "Brier": brier,
        "Optimal_Threshold": best_th,
        "Val_F1": best_f1,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "test_probs": test_probs,
    }

results = [
    evaluate_model_pipeline("Logistic Regression", val_p_lr, test_p_lr, y_val, y_test),
    evaluate_model_pipeline("LightGBM",            val_p_lgb, test_p_lgb, y_val, y_test),
    evaluate_model_pipeline("XGBoost",             val_p_xgb, test_p_xgb, y_val, y_test),
    evaluate_model_pipeline("Stacked Ensemble",    val_p_ensemble, test_p_ensemble, y_val, y_test),
]

res_df = pd.DataFrame(results).drop(columns=["test_probs"])

# ── Sắp xếp theo tiêu chí: AUCPR giảm dần, sau đó ROC_AUC giảm dần ──
res_sorted = res_df.sort_values(by=["AUCPR", "ROC_AUC"], ascending=[False, False]).reset_index(drop=True)

print("=" * 92)
print("              BẢNG SO SÁNH HIỆU NĂNG 4 MÔ HÌNH TRÊN TẬP TEST ĐỘC LẬP")
print("=" * 92)
print(f"  {'Hạng':<5} {'Mô hình':<22} {'AUCPR':>10} {'ROC-AUC':>10} {'F1-Score':>10} {'Precision':>11} {'Recall':>10} {'Ngưỡng':>8}")
print("-" * 92)
for idx, r in res_sorted.iterrows():
    star = " ⭐ (BEST)" if idx == 0 else ""
    print(f"  #{idx+1:<4} {r['Model']:<22} {r['AUCPR']:>10.4f} {r['ROC_AUC']:>10.4f} {r['F1']:>10.4f} {r['Precision']:>10.2%} {r['Recall']:>10.2%} {r['Optimal_Threshold']:>8.2f}{star}")
print("=" * 92)

best_model_name = res_sorted.iloc[0]["Model"]
best_aucpr = res_sorted.iloc[0]["AUCPR"]
best_roc_auc = res_sorted.iloc[0]["ROC_AUC"]

print(f"\n🏆 MÔ HÌNH CHIẾN THẮNG ĐƯỢC CHỌN: {best_model_name}")
print(f"   - Test AUCPR   : {best_aucpr:.4f}")
print(f"   - Test ROC-AUC : {best_roc_auc:.4f}")
print(f"   - Test F1      : {res_sorted.iloc[0]['F1']:.4f} (ở ngưỡng {res_sorted.iloc[0]['Optimal_Threshold']:.2f})")

# Lưu bảng so sánh
res_sorted.to_csv(OUTPUT_DIR / "ensemble_model_comparison_v2.csv", index=False)
print(f"\nĐã lưu bảng so sánh: {OUTPUT_DIR / 'ensemble_model_comparison_v2.csv'}")


              BẢNG SO SÁNH HIỆU NĂNG 4 MÔ HÌNH TRÊN TẬP TEST ĐỘC LẬP
  Hạng  Mô hình                     AUCPR    ROC-AUC   F1-Score   Precision     Recall   Ngưỡng
--------------------------------------------------------------------------------------------
  #1    LightGBM                   0.5567     0.9543     0.6878     54.96%     91.91%     0.53 ⭐ (BEST)
  #2    Stacked Ensemble           0.5565     0.9542     0.6907     55.03%     92.71%     0.27
  #3    XGBoost                    0.5542     0.9541     0.6917     55.19%     92.66%     0.54
  #4    Logistic Regression        0.5454     0.9478     0.6854     54.87%     91.28%     0.29

🏆 MÔ HÌNH CHIẾN THẮNG ĐƯỢC CHỌN: LightGBM
   - Test AUCPR   : 0.5567
   - Test ROC-AUC : 0.9543
   - Test F1      : 0.6878 (ở ngưỡng 0.53)

Đã lưu bảng so sánh: D:\Intern Data\output\ensemble_model_comparison_v2.csv


## 10. Trực quan hóa So sánh ROC Curve & PR Curve


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = {
    "Logistic Regression": "#7f8c8d",
    "LightGBM": "#2ecc71",
    "XGBoost": "#3498db",
    "Stacked Ensemble": "#e74c3c",
}

# ── 1. ROC Curves ──────────────────────────────────────────────────
ax1 = axes[0]
for r in results:
    fpr, tpr, _ = roc_curve(y_test, r["test_probs"])
    lw = 3 if r["Model"] == best_model_name else 1.8
    ax1.plot(fpr, tpr, color=colors[r["Model"]], lw=lw,
             label=f"{r['Model']} (AUC = {r['ROC_AUC']:.4f})")

ax1.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random (AUC = 0.5)")
ax1.set_xlabel("False Positive Rate", fontsize=11)
ax1.set_ylabel("True Positive Rate", fontsize=11)
ax1.set_title("So sánh ROC Curves trên Tập Test", fontsize=13, fontweight="bold")
ax1.legend(loc="lower right", fontsize=9)
ax1.grid(True, alpha=0.3)

# ── 2. Precision-Recall Curves ────────────────────────────────────
ax2 = axes[1]
for r in results:
    prec, rec, _ = precision_recall_curve(y_test, r["test_probs"])
    lw = 3 if r["Model"] == best_model_name else 1.8
    ax2.plot(rec, prec, color=colors[r["Model"]], lw=lw,
             label=f"{r['Model']} (AUCPR = {r['AUCPR']:.4f})")

base_rate = y_test.mean()
ax2.axhline(y=base_rate, color="gray", linestyle="--", lw=1, alpha=0.5,
            label=f"Baseline ({base_rate:.2%})")
ax2.set_xlabel("Recall", fontsize=11)
ax2.set_ylabel("Precision", fontsize=11)
ax2.set_title("So sánh Precision-Recall Curves trên Tập Test", fontsize=13, fontweight="bold")
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
chart_path = OUTPUT_DIR / "ensemble_comparison_curves.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Đã lưu biểu đồ: {chart_path}")


Đã lưu biểu đồ: D:\Intern Data\output\ensemble_comparison_curves.png


## 11. Phân tích Tầm quan trọng Đặc trưng (Variable Importance — GBM)
Trích xuất và trực quan hóa Top các đặc trưng đóng góp lớn nhất vào quyết định phân loại của mô hình LightGBM.


In [11]:
# Trích xuất Feature Importance từ mô hình LightGBM đã retrain
fi_series = pd.Series(model_lgb.feature_importances_, index=feature_cols)
fi_df = fi_series.sort_values(ascending=False).reset_index()
fi_df.columns = ["feature", "importance_split_gain"]
fi_df["rank"] = range(1, len(fi_df) + 1)

# Lưu file CSV
fi_csv_path = OUTPUT_DIR / "feature_importance_ensemble_v2.csv"
fi_df.to_csv(fi_csv_path, index=False)

print("=" * 65)
print("TOP 20 ĐẶC TRƯNG QUAN TRỌNG NHẤT (LIGHTGBM SPLIT GAIN)")
print("=" * 65)
print(f"  {'Hạng':>4}  {'Tên đặc trưng':<45} {'Điểm Gain':>10}")
print("-" * 65)
for _, row in fi_df.head(20).iterrows():
    print(f"  {int(row['rank']):>4}  {row['feature']:<45} {row['importance_split_gain']:>10.0f}")
print("=" * 65)

# Vẽ biểu đồ Barplot Top 25
top_k = 25
top_fi = fi_df.head(top_k).iloc[::-1]

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top_fi["feature"], top_fi["importance_split_gain"], color="#2ecc71", edgecolor="black", alpha=0.85)
ax.set_xlabel("Split Importance Gain", fontsize=11)
ax.set_title(f"Top {top_k} Đặc Trưng Quan Trọng Nhất (LightGBM)", fontsize=13, fontweight="bold")
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
fi_chart_path = OUTPUT_DIR / "feature_importance_top25.png"
plt.savefig(fi_chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Đã lưu biểu đồ đặc trưng: {fi_chart_path}")


TOP 20 ĐẶC TRƯNG QUAN TRỌNG NHẤT (LIGHTGBM SPLIT GAIN)
  Hạng  Tên đặc trưng                                  Điểm Gain
-----------------------------------------------------------------
     1  days_since_last_support_ticket                       235
     2  payment_success_rolling_mean_6m                      228
     3  days_since_last_payment                              177
     4  days_since_last_usage                                176
     5  orders_rolling_std_6m                                171
     6  marketing_interaction_rolling_std_6m                 160
     7  payment_count_rolling_std_6m                         155
     8  days_since_last_order                                140
     9  payment_success_rate_rolling_sum_6m                  140
    10  spend_rolling_mean_6m                                134
    11  spend_rolling_std_6m                                 134
    12  orders_rolling_mean_6m                               118
    13  payment_success_rate_rolli

## 12. Đóng gói & Lưu trữ Model Bundle


In [12]:
# Đóng gói toàn bộ Pipeline và Artifacts
bundle = {
    "pipeline_type": "Stacked_Ensemble_v2",
    "business_rule_version": "v2",
    "best_model_name": best_model_name,
    "feature_names": feature_cols,
    "feature_count": len(feature_cols),
    
    # ── Các thành phần mô hình ──
    "imputer": imputer,
    "scaler": scaler,
    "base_model_lr": model_lr,
    "base_model_lgb": model_lgb,
    "base_model_xgb": model_xgb,
    "meta_learner": meta_learner,
    
    # ── Tham số & Ngưỡng tối ưu ──
    "best_lgb_params": best_lgb_params,
    "optimal_thresholds": {r["Model"]: r["Optimal_Threshold"] for r in results},
    "selected_threshold": res_sorted.iloc[0]["Optimal_Threshold"],
    
    # ── Kết quả đánh giá ──
    "comparison_metrics": res_sorted.to_dict(orient="records"),
    "created_at": datetime.datetime.now().isoformat(),
}

bundle_path = ART_DIR / "advanced_ensemble_churn_v2.joblib"
joblib.dump(bundle, bundle_path)

print("=" * 65)
print("ĐÓNG GÓI VÀ LƯU TRỮ MÔ HÌNH THÀNH CÔNG")
print("=" * 65)
print(f"  Vị trí lưu trữ      : {bundle_path.resolve()}")
print(f"  Kích thước file     : {bundle_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Mô hình được chọn   : {bundle['best_model_name']}")
print(f"  Ngưỡng phân loại    : {bundle['selected_threshold']:.2f}")
print(f"  Số lượng đặc trưng  : {bundle['feature_count']}")
print("=" * 65)

# ── Kiểm tra khả năng Tải lại và Dự đoán mẫu ─────────────────────────
loaded = joblib.load(bundle_path)
sample_X = X_test_raw.iloc[:5]
sample_imp = loaded["imputer"].transform(sample_X)
sample_sc  = loaded["scaler"].transform(sample_imp)

p_lr_s  = loaded["base_model_lr"].predict_proba(sample_sc)[:, 1]
p_lgb_s = loaded["base_model_lgb"].predict_proba(sample_imp)[:, 1]
p_xgb_s = loaded["base_model_xgb"].predict_proba(sample_imp)[:, 1]
p_ens_s = loaded["meta_learner"].predict_proba(np.column_stack([p_lr_s, p_lgb_s, p_xgb_s]))[:, 1]
preds_s = (p_ens_s >= loaded["selected_threshold"]).astype(int)

print(f"Kiểm tra tải lại Bundle & Dự đoán 5 mẫu: THÀNH CÔNG (PASS)")
print(f"Xác suất dự đoán Ensemble: {np.round(p_ens_s, 4)}")
print(f"Nhãn phân loại cuối cùng : {preds_s.tolist()}")
print("=" * 65)


ĐÓNG GÓI VÀ LƯU TRỮ MÔ HÌNH THÀNH CÔNG
  Vị trí lưu trữ      : D:\Intern Data\artifacts\advanced_ensemble_churn_v2.joblib
  Kích thước file     : 2.15 MB
  Mô hình được chọn   : LightGBM
  Ngưỡng phân loại    : 0.53
  Số lượng đặc trưng  : 379
Kiểm tra tải lại Bundle & Dự đoán 5 mẫu: THÀNH CÔNG (PASS)
Xác suất dự đoán Ensemble: [0.0192 0.0192 0.0196 0.0198 0.6171]
Nhãn phân loại cuối cùng : [0, 0, 0, 0, 1]


## 13. Tổng kết & Đề xuất Triển khai Thực tế

### 📊 Bảng so sánh 4 mô hình:
- **Logistic Regression**: Baseline vững chắc, đóng vai trò tạo xác suất chuẩn hóa tuyến tính.
- **LightGBM**: Tốc độ huấn luyện siêu nhanh, tối ưu hóa theo AUCPR cho hiệu năng cực cao.
- **XGBoost**: Cung cấp đa dạng hóa góc nhìn phân chia cây quyết định với kiểm soát Regularization L1/L2.
- **Stacked Ensemble**: Kết hợp tối ưu điểm mạnh của cả 3 mô hình, giúp đường cong PR-AUC ổn định và tăng độ bao phủ (Recall) ở các phân khúc khách hàng khó phát hiện.

### 🎯 Chiến lược triển khai:
1. Sử dụng file bundle `artifacts/advanced_ensemble_churn_v2.joblib` trong Inference Pipeline hàng tháng.
2. Thiết lập hệ thống giám sát (Monitoring) độ lệch dữ liệu (Data Drift) và độ lệch dự đoán (Prediction Drift) dựa trên phân phối xác suất đầu ra.
